# SYNTHETIC DATA GENERATOR

**Project: Utilization Pattern Monitoring**  

IMPORTANT:
- No original data is used.
- No statistics derived from original data are used.
- No real-world identifiers are used.
- All distributions and relationships are fully synthetic.
- The random seed and the patient count are the only configurable metadata parameters.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# 1 DATASET METADATA

In [2]:
SEED = 20260917
N_PACIENTES = 100_000

# 2 RANDOM NUMBER GENERATOR CONFIGURATION

In [3]:
rng = np.random.default_rng(SEED)

# 3 SYNTHETIC IDENTIFIERS

In [4]:
pacientes = [
    f"SYN_{i:06d}"
    for i in range(1, N_PACIENTES + 1)
]

# 4 NUMBER OF VISITS

Most patients are assumed to have relatively few hospital visits.  
A small number of patients are assumed to have a high frequency of hospital visits.  

>These parameters are not based on or derived from any real-world data source.

In [5]:
# SYNTHETIC GEOMETRIC DISTRRIBUTION
num_visitas = rng.geometric(
    p=0.50,
    size=N_PACIENTES
)

# Extremely high values are capped to maintain a reasonable range
num_visitas = np.clip(
    num_visitas,
    1,
    50
)

# 5 NUMBER OF MEDICAL SPECIALTIES

The number of specialties is partially influenced by the number of episodes, but the relationship is entirely synthetic.  

The rationale is:  
- Patients with few episodes typically have a small number of specialties.
- As the number of episodes increases, the likelihood of having more specialties also increases.

In [6]:
# Synthetic probability of having at least one medical specialty
prob_especialidad = (
    0.35
    + 0.08 * np.log1p(num_visitas)
)

prob_especialidad = np.clip(
    prob_especialidad,
    0.35,
    0.85
)

tiene_especialidad = (
    rng.random(N_PACIENTES)
    < prob_especialidad
)

# Base number of medical specialties
num_especialidades = np.zeros(N_PACIENTES, dtype=int)

# For patients with at least one medical specialty
idx = tiene_especialidad

num_especialidades[idx] = (
    1
    + rng.poisson(
        lam=0.35 + 0.12 * np.log1p(num_visitas[idx]),
        size=idx.sum()
    )
)

num_especialidades = np.clip(
    num_especialidades,
    0,
    10
)

# 6 NUMBER OF DIAGNOSES

Synthetic relationship:  

- An increasing number of episodes may lead to more opportunities for diagnosis recording.
- The majority of patients may still have no recorded diagnoses.

In [7]:
# Synthetic probability of having recorded diagnoses
prob_diagnostico = (
    0.25
    + 0.10 * np.log1p(num_visitas)
    + 0.04 * num_especialidades
)

prob_diagnostico = np.clip(
    prob_diagnostico,
    0.25,
    0.90
)

tiene_diagnostico = (
    rng.random(N_PACIENTES)
    < prob_diagnostico
)

num_diagnosticos = np.zeros(
    N_PACIENTES,
    dtype=int
)

idx = tiene_diagnostico

num_diagnosticos[idx] = (
    1
    + rng.poisson(
        lam=0.30
        + 0.10 * np.log1p(num_visitas[idx])
        + 0.10 * num_especialidades[idx],
        size=idx.sum()
    )
)

num_diagnosticos = np.clip(
    num_diagnosticos,
    0,
    20
)

# 7 HOSPITALIZATION RATE

Generated as a proportional variable ranging from 0 to 1.  

Three synthetic states are used:  
0: no hospitalization  
partial: a mix of hospitalized and non-hospitalized visits  
1: all visits are hospitalized  

>The probabilities are based on SYNTHETIC ASSUMPTIONS.

In [8]:
pct_hospitalizacion = np.zeros(
    N_PACIENTES,
    dtype=float
)

# Synthetic probability of belonging to a group with some degree of hospitalization
prob_hospitalizacion = (
    0.08
    + 0.025 * np.log1p(num_visitas)
    + 0.015 * num_especialidades
)

prob_hospitalizacion = np.clip(
    prob_hospitalizacion,
    0.08,
    0.75
)

tiene_hospitalizacion = (
    rng.random(N_PACIENTES)
    < prob_hospitalizacion
)

# Patients with hospitalization
idx = tiene_hospitalizacion

# A synthetic proportion is generated using a Beda distribution
pct_hospitalizacion[idx] = rng.beta(
    a=2.0,
    b=2.0,
    size=idx.sum()
)

# Some patients will have complete hospitalization
# Synthetic probability
hospitalizacion_total = (
    idx
    & (
        rng.random(N_PACIENTES)
        < 0.20
    )
)

pct_hospitalizacion[hospitalizacion_total] = 1.0

# 8 DATASET GENERATION

In [9]:
df_sintetico = pd.DataFrame({
    "Paciente": pacientes,
    "num_visitas": num_visitas.astype(int),
    "Num_Especialidades": num_especialidades.astype(int),
    "Num_Diagnosticos": num_diagnosticos.astype(int),
    "Pct_Hospitalizacion": pct_hospitalizacion
})


# 9 VALUE ROUNDING

In [10]:
df_sintetico["Pct_Hospitalizacion"] = (
    df_sintetico["Pct_Hospitalizacion"]
    .round(4)
)

# 10 DATA VALIDATIONS

In [11]:
assert len(df_sintetico) == N_PACIENTES

assert df_sintetico["Paciente"].is_unique

assert df_sintetico["num_visitas"].ge(1).all()

assert df_sintetico["Num_Especialidades"].ge(0).all()

assert df_sintetico["Num_Diagnosticos"].ge(0).all()

assert df_sintetico["Pct_Hospitalizacion"].between(
    0,
    1
).all()

assert not df_sintetico.isna().any().any()

# 11 DATASET INFORMATION

In [12]:
print("=" * 70)
print("DATASET SINTÉTICO")
print("=" * 70)

print(f"Semilla:       {SEED}")
print(f"Pacientes:     {N_PACIENTES:,}")
print(f"Registros:     {len(df_sintetico):,}")

print("\nColumnas:")
print(df_sintetico.columns.tolist())

print("\nTipos de datos:")
print(df_sintetico.dtypes)

print("\nPrimeros registros:")
print(df_sintetico.head())

print("\nEstadísticas:")
print(df_sintetico.describe())

DATASET SINTÉTICO
Semilla:       20260917
Pacientes:     100,000
Registros:     100,000

Columnas:
['Paciente', 'num_visitas', 'Num_Especialidades', 'Num_Diagnosticos', 'Pct_Hospitalizacion']

Tipos de datos:
Paciente                   str
num_visitas              int64
Num_Especialidades       int64
Num_Diagnosticos         int64
Pct_Hospitalizacion    float64
dtype: object

Primeros registros:
     Paciente  num_visitas  Num_Especialidades  Num_Diagnosticos  \
0  SYN_000001            2                   1                 0   
1  SYN_000002            1                   0                 0   
2  SYN_000003            1                   1                 0   
3  SYN_000004            2                   0                 1   
4  SYN_000005            1                   0                 0   

   Pct_Hospitalizacion  
0               0.0000  
1               0.0000  
2               0.0000  
3               0.0000  
4               0.5246  

Estadísticas:
         num_visitas  Num_E

# 12 VARIABLE CORRELATIONS

In [13]:
print("\n" + "=" * 70)
print("CORRELACIONES")
print("=" * 70)

print(
    df_sintetico[
        [
            "num_visitas",
            "Num_Especialidades",
            "Num_Diagnosticos",
            "Pct_Hospitalizacion"
        ]
    ].corr().round(3)
)


CORRELACIONES
                     num_visitas  Num_Especialidades  Num_Diagnosticos  \
num_visitas                1.000               0.081             0.095   
Num_Especialidades         0.081               1.000             0.110   
Num_Diagnosticos           0.095               0.110             1.000   
Pct_Hospitalizacion        0.032               0.037             0.006   

                     Pct_Hospitalizacion  
num_visitas                        0.032  
Num_Especialidades                 0.037  
Num_Diagnosticos                   0.006  
Pct_Hospitalizacion                1.000  


# 13 DATA EXPORT

In [14]:
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "pacientes_sinteticos.csv"
)

df_sintetico.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print("\n" + "=" * 70)
print(f"Archivo generado: {OUTPUT_FILE}")
print("=" * 70)


Archivo generado: data\pacientes_sinteticos.csv
